# RDataLoader: Native ROOT Data Loading for Machine Learning

## 🎯 Learning Objectives

At the beginning of this course you should already be familiar with:
* Creating and querying an `RDataFrame`
* Basic PyTorch training loops

By the end of this notebook, you will be able to:

* **Build batches** of training/validation data directly from an ROOT data, with no intermediate file conversion
* **Feed those batches** into NumPy, PyTorch and Tensorflow workflows

## ✅ Environment Check

`RDataLoader` lives under `ROOT.Experimental.ML` and is under active development, so its API can change between ROOT versions.

In [ ]:
import ROOT

print(f"ROOT version: {ROOT.gROOT.GetVersion()}")

assert ROOT.gROOT.GetVersionInt() >= 64000, (
    "ROOT.Experimental.ML.RDataLoader is not available in this build. "
    "ROOT >= 6.40 is required."
)

---
## 🧱 Why RDataLoader?

Training an ML model on data stored in a `TTree` or an `RNTuple` usually means either:

- writing a custom batch generator too feed the ROOT data to ML models, or
- converting ROOT data to other data formats like parquet or HDF5

`RDataLoader` builds batches **directly from an `RDataFrame`**, so you get RDataFrame's capabilities, and the batches come out as whatever your ML framework expects (NumPy arrays, PyTorch tensors, TensorFlow Datasets, or JAX arrays) with no manual conversion step.

> Think of it as the ROOT-native equivalent of a PyTorch `DataLoader`, for ROOT data.

<center><img src="../images/motivation.png"></center>

---
## 🔢 Basic Usage: Batches as NumPy Arrays

We'll use the Higgs classification dataset that ships with ROOT's tutorials. `RDataLoader` takes an `RDataFrame`, a `batch_size`, and the name of the `target` column (the label).

In [ ]:
rdataframe = ROOT.RDataFrame("sig_tree", "../data/Higgs_data.root")

rdataframe.Display()

In [ ]:
batch_size = 128
target = "Type"

dl = ROOT.Experimental.ML.RDataLoader(
    rdataframe,
    batch_size,
    target=target,
    shuffle=True,
    drop_remainder=True,
)

# train_test_split gives back two RDataLoader "views", each referencing a fraction of the original dataset
gen_train, gen_validation = dl.train_test_split(test_size=0.3)

In [ ]:
num_of_epochs = 1

for epoch in range(num_of_epochs):
    for i, (x_train, y_train) in enumerate(gen_train.as_numpy()):
        if i < 3:
            print(f"Training batch {i + 1} => x: {x_train.shape}, y: {y_train.shape}")

    for i, (x_val, y_val) in enumerate(gen_validation.as_numpy()):
        if i < 3:
            print(f"Validation batch {i + 1} => x: {x_val.shape}, y: {y_val.shape}")

### 🔑 Key `RDataLoader` Parameters

| Parameter | Meaning |
|---:|:---|
| `batch_size` | Number of rows per batch |
| `target` | Name of the label column. |
| `columns` | Explicit list of columns to load. If omitted, all columns are used |
| `shuffle` | Shuffle rows before batching |
| `drop_remainder` | Drop the final incomplete batch instead of yielding a short one |
| `set_seed` | Seed for the shuffling / sampling RNG, for reproducibility |
| `batches_in_memory` | Number of batches prefetched at once |
| `max_vec_sizes` | Required whenever a column holds a jagged/vector value (e.g. `RVec`); maps column name → fixed size to pad/truncate to |

### 💡 Framework Cheat Sheet

| **Call** | **Returns** |
|---:|:---|
| `dl.as_numpy()` | generator of `(x, y)` NumPy array batches |
| `dl.as_torch()` | generator of `(x, y)` PyTorch tensor batches |
| `dl.as_tf()` | `(x, y)` TensorFlow Dataset object |
| `dl.train_test_split(test_size=...)` | two `RDataLoader` views: train, validation |

See [RDataLoader extended Cheat Sheet](https://root.cern/doc/master/group__Py__ML.html)

---
## 🔥 Feeding a PyTorch Model

The same `RDataLoader` view works as a drop-in batch source for a normal PyTorch training loop, just swap `.as_numpy()` for `.as_torch()`.

In [ ]:
import torch

input_columns = gen_train.train_columns
num_features = len(input_columns)

model = torch.nn.Sequential(
    torch.nn.Linear(num_features, 64),
    torch.nn.Tanh(),
    torch.nn.Linear(64, 1),
    torch.nn.Sigmoid(),
)

loss_fn = torch.nn.MSELoss(reduction="mean")
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)


def calc_accuracy(targets, pred):
    return torch.sum(targets == pred.round()) / pred.size(0)


model.train()
for i, (x_train, y_train) in enumerate(gen_train.as_torch()):
    pred = model(x_train)
    loss = loss_fn(pred, y_train)

    model.zero_grad()
    loss.backward()
    optimizer.step()

    if i < 3:
        accuracy = calc_accuracy(y_train, pred)
        print(f"Training batch {i + 1} => accuracy: {accuracy:.3f}")

---
## 📦 Jagged (Vector) Columns

Some datasets store per-event vectors of varying length (e.g. per-lepton quantities in an `RVec<float>` column). `RDataLoader` needs a **fixed** size to build rectangular batches, so any such column must be listed in `max_vec_sizes`:

```python
columns = ["m4l", "goodlep_pt", "goodlep_eta", "isHiggsRef"]
target = "isHiggsRef"
max_vec_sizes = {"goodlep_pt": 4, "goodlep_eta": 4}  # pad/truncate to 4 leptons

dl = ROOT.Experimental.ML.RDataLoader(
    rdataframe,
    batch_size=1000,
    columns=columns,
    target=target,
    max_vec_sizes=max_vec_sizes,
)
```

> ⚠️ Each vector column in `max_vec_sizes` is expanded into `max_vec_sizes[col]` scalar features. When computing `num_features` for your model input layer by hand, remember `train_columns` still lists the *condensed* vector column name, not its expanded components.

---
## ⚖️ Class Imbalance: Over- and Undersampling

If one class is underrepresented, `RDataLoader` can resample it while building batches. This requires `load_eager=True`, and you pass a list of `RDataFrame`s (one per class) instead of a single one.

In [ ]:
# df_major: well-represented class, df_minor: underrepresented class
df_major = ROOT.RDataFrame(100000).Define("b1", "(int) 2 * rdfentry_").Define("b2", "(int) b1 % 2")
df_minor = ROOT.RDataFrame(1000).Define("b1", "(int) 2 * rdfentry_ + 1").Define("b2", "(int) b1 % 2")

dl_oversampled = ROOT.Experimental.ML.RDataLoader(
    [df_major, df_minor],
    batch_size=256,
    target="b2",
    set_seed=42,
    load_eager=True,  # required for resampling
    sampling_type="oversampling",  # or "undersampling"
    sampling_ratio=0.1,  # minority class will make up ~10% of each batch
)

### 💡 Sampling Cheat Sheet

| **Parameter** | **Effect** |
|---:|:---|
| `sampling_type="oversampling"` | Draws the minority class more often; more batches per epoch, more balanced training |
| `sampling_type="undersampling"` | Draws the majority class less often; fewer batches per epoch, faster but discards data |
| `sampling_ratio` | Target fraction of the minority class per batch |
| `load_eager=True` | Required for either sampling mode to work |

---
## 📚 Further Reading

### Reference
- [RDataLoader](https://root.cern/doc/master/group__Py__ML.html)

### Tutorials
- [ml_dataloader_Higgs_Classification.py](https://root.cern.ch/doc/master/ml__dataloader__Higgs__Classification_8py.html)
- [ml_dataloader_resampling.py](https://root.cern.ch/doc/master/ml__dataloader__resampling_8py.html)